# Fake News Detection — Multi-Agent ML Pipeline

In [ ]:
import os
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from datasets import load_dataset
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
)
from anthropic import Anthropic

In [ ]:

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 120)

In [ ]:
try:
    from google.colab import userdata
    _api_key = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    _api_key = os.getenv("ANTHROPIC_API_KEY")

if not _api_key:
    raise EnvironmentError(
        "ANTHROPIC_API_KEY not found. "
        "Set it in Colab userdata or as an environment variable."
    )

os.environ["ANTHROPIC_API_KEY"] = _api_key
anthropic_client = Anthropic(api_key=_api_key)

In [ ]:
# Base Agent
class BaseAgent:
    """
    Minimal base class for all agents.
    Every agent has a name, a run() entry-point, and a _log() helper.
    """
    def __init__(self, name: str):
        self.name = name

    def run(self, *args, **kwargs):
        raise NotImplementedError(f"{self.name}.run() must be implemented.")

    def _log(self, msg: str):
        print(f"[{self.name}] {msg}")

# IngestionAgent  — load & clean raw data
class IngestionAgent(BaseAgent):
    """
    Downloads the fake-news dataset from HuggingFace, merges
    headline + body text, assigns binary labels, and does
    basic cleaning.
    """

    DATASET_ID = "GonzaloA/fake_news"
    SAMPLE_SIZE = 6_000   # cap to keep runtime manageable

    def __init__(self):
        super().__init__("IngestionAgent")

    def _clean(self, text: str) -> str:
        if not isinstance(text, str):
            return ""
        text = text.lower()
        text = re.sub(r"http\S+|www\S+", " ", text)          # strip URLs
        text = re.sub(r"[^a-z0-9\s]", " ", text)             # keep alphanum
        text = re.sub(r"\s+", " ", text).strip()
        return text

    def run(self) -> pd.DataFrame:
        self._log(f"Loading '{self.DATASET_ID}' from HuggingFace…")
        raw = load_dataset(self.DATASET_ID, split="train")
        df  = pd.DataFrame(raw)

        # Keep only what we need; combine title + text for richer signal
        df = df[["title", "text", "label"]].dropna().reset_index(drop=True)
        df["combined"] = (df["title"].fillna("") + " " + df["text"].fillna(""))
        df["combined"] = df["combined"].apply(self._clean)
        df["label"]    = df["label"].astype(int)   # 0 = FAKE, 1 = REAL

        # Cap sample size
        if len(df) > self.SAMPLE_SIZE:
            df = df.sample(self.SAMPLE_SIZE, random_state=7).reset_index(drop=True)

        df["word_count"] = df["combined"].apply(lambda x: len(x.split()))
        self._log(
            f"Ready: {len(df):,} articles  |  "
            f"FAKE={( df.label==0).sum():,}  REAL={(df.label==1).sum():,}"
        )
        return df

#  FeatureAgent  — EDA + feature engineering
class FeatureAgent(BaseAgent):
    """
    Produces exploratory charts and enriches the DataFrame
    with hand-crafted linguistic features that complement TF-IDF.
    """

    def __init__(self):
        super().__init__("FeatureAgent")

    # Hand-crafted features
    @staticmethod
    def _exclamation_density(text: str) -> float:
        words = text.split()
        if not words:
            return 0.0
        return text.count("!") / len(words)

    @staticmethod
    def _question_density(text: str) -> float:
        words = text.split()
        if not words:
            return 0.0
        return text.count("?") / len(words)

    @staticmethod
    def _capital_ratio(text: str) -> float:
        letters = [c for c in text if c.isalpha()]
        if not letters:
            return 0.0
        return sum(1 for c in letters if c.isupper()) / len(letters)

    # EDA plots 
    def _plot_eda(self, df: pd.DataFrame):
        label_names = {0: "Fake", 1: "Real"}
        df["label_name"] = df["label"].map(label_names)

        fig, axes = plt.subplots(1, 3, figsize=(16, 4))
        fig.suptitle("Exploratory Analysis — Fake vs Real News", fontsize=14, fontweight="bold")

        # 1. Class balance
        counts = df["label_name"].value_counts()
        axes[0].bar(counts.index, counts.values, color=["#e05252", "#52a8e0"], edgecolor="white", width=0.5)
        axes[0].set_title("Class Distribution")
        axes[0].set_ylabel("Article Count")
        for i, v in enumerate(counts.values):
            axes[0].text(i, v + 20, str(v), ha="center", fontweight="bold")

        # 2. Word count distributions
        for lbl, colour in zip([0, 1], ["#e05252", "#52a8e0"]):
            subset = df[df["label"] == lbl]["word_count"]
            axes[1].hist(subset, bins=40, alpha=0.6, color=colour,
                         label=label_names[lbl], edgecolor="none")
        axes[1].set_title("Article Length Distribution")
        axes[1].set_xlabel("Word Count")
        axes[1].legend()

        # 3. Exclamation density boxplot
        sns.boxplot(
            data=df, x="label_name", y="exclamation_density",
            palette={"Fake": "#e05252", "Real": "#52a8e0"},
            ax=axes[2], width=0.4,
        )
        axes[2].set_title("Exclamation Density by Class")
        axes[2].set_xlabel("")
        axes[2].set_ylabel("Exclamations per Word")

        plt.tight_layout()
        plt.show()

    def run(self, df: pd.DataFrame) -> pd.DataFrame:
        self._log("Engineering features…")
        df = df.copy()
        df["exclamation_density"] = df["combined"].apply(self._exclamation_density)
        df["question_density"]    = df["combined"].apply(self._question_density)
        df["capital_ratio"]       = df["combined"].apply(self._capital_ratio)

        self._log("Rendering EDA charts…")
        self._plot_eda(df)
        self._log("Feature engineering complete.")
        return df

# ClassifierAgent  — train, evaluate, and visualise
class ClassifierAgent(BaseAgent):
    """
    Trains a soft-voting ensemble (Logistic Regression + Random Forest),
    evaluates on a held-out test set, and returns a metrics dict.
    """

    TEST_RATIO  = 0.20
    RANDOM_SEED = 42

    def __init__(self):
        super().__init__("ClassifierAgent")
        self.model     = None
        self.vectorizer = None

    # Build pipelines for each base learner
    def _build_ensemble(self):
        lr_pipe = Pipeline([
            ("tfidf", TfidfVectorizer(max_features=15_000, ngram_range=(1, 2),
                                      sublinear_tf=True, min_df=3)),
            ("clf",   LogisticRegression(max_iter=500, C=1.0,
                                          class_weight="balanced", random_state=self.RANDOM_SEED)),
        ])
        rf_pipe = Pipeline([
            ("tfidf", TfidfVectorizer(max_features=10_000, ngram_range=(1, 2),
                                      sublinear_tf=True, min_df=3)),
            ("clf",   RandomForestClassifier(n_estimators=150, max_depth=20,
                                              class_weight="balanced",
                                              random_state=self.RANDOM_SEED, n_jobs=-1)),
        ])
        ensemble = VotingClassifier(
            estimators=[("lr", lr_pipe), ("rf", rf_pipe)],
            voting="soft",
        )
        return ensemble

    def _plot_results(self, y_test, y_pred, y_proba, label_names):
        fig, axes = plt.subplots(1, 2, figsize=(13, 5))
        fig.suptitle("Classifier Evaluation", fontsize=14, fontweight="bold")

        # Confusion matrix
        cm = confusion_matrix(y_test, y_pred)
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                    xticklabels=label_names, yticklabels=label_names, ax=axes[0])
        axes[0].set_title("Confusion Matrix")
        axes[0].set_xlabel("Predicted")
        axes[0].set_ylabel("Actual")

        # ROC curve
        fpr, tpr, _ = roc_curve(y_test, y_proba)
        auc_score   = roc_auc_score(y_test, y_proba)
        axes[1].plot(fpr, tpr, lw=2, color="#3a86ff",
                     label=f"ROC  (AUC = {auc_score:.3f})")
        axes[1].plot([0, 1], [0, 1], "--", color="grey", lw=1)
        axes[1].set_title("ROC Curve")
        axes[1].set_xlabel("False Positive Rate")
        axes[1].set_ylabel("True Positive Rate")
        axes[1].legend()
        axes[1].grid(alpha=0.3)

        plt.tight_layout()
        plt.show()

    def run(self, df: pd.DataFrame) -> dict:
        self._log("Splitting data…")
        X = df["combined"]
        y = df["label"]

        X_train, X_test, y_train, y_test = train_test_split(
            X, y,
            test_size=self.TEST_RATIO,
            stratify=y,
            random_state=self.RANDOM_SEED,
        )

        self._log("Training ensemble (LR + RF voting classifier)…")
        ensemble = self._build_ensemble()
        ensemble.fit(X_train, y_train)
        self.model = ensemble

        y_pred  = ensemble.predict(X_test)
        y_proba = ensemble.predict_proba(X_test)[:, 1]
        auc     = roc_auc_score(y_test, y_proba)

        report_dict = classification_report(y_test, y_pred,
                                            target_names=["Fake", "Real"],
                                            output_dict=True)

        self._log("\n" + classification_report(y_test, y_pred,
                                               target_names=["Fake", "Real"]))
        self._log(f"ROC-AUC: {auc:.4f}")

        self._plot_results(y_test, y_pred, y_proba, ["Fake", "Real"])

        metrics = {
            "accuracy":      report_dict["accuracy"],
            "fake_f1":       report_dict["Fake"]["f1-score"],
            "real_f1":       report_dict["Real"]["f1-score"],
            "macro_f1":      report_dict["macro avg"]["f1-score"],
            "roc_auc":       auc,
            "train_samples": len(X_train),
            "test_samples":  len(X_test),
        }
        return metrics
    
# InsightAgent  — LLM-powered narrative summary
class InsightAgent(BaseAgent):
    """
    Sends the classification metrics to Claude and asks for a
    plain-English interpretation suitable for a non-technical audience.
    """

    MODEL_ID   = "claude-3-5-haiku-20241022"
    MAX_TOKENS = 350

    def __init__(self, client: Anthropic):
        super().__init__("InsightAgent")
        self.client = client

    def run(self, metrics: dict) -> str:
        self._log("Requesting narrative summary from Claude…")

        prompt = f"""
You are a data science communicator writing for a non-technical business audience.

A machine-learning ensemble model was trained to automatically distinguish
REAL news articles from FAKE ones. Here are the results on the held-out test set:

  - Overall Accuracy : {metrics['accuracy']*100:.1f}%
  - Fake-News F1     : {metrics['fake_f1']:.3f}
  - Real-News F1     : {metrics['real_f1']:.3f}
  - Macro-Avg F1     : {metrics['macro_f1']:.3f}
  - ROC-AUC Score    : {metrics['roc_auc']:.3f}
  - Training samples : {metrics['train_samples']:,}
  - Test samples     : {metrics['test_samples']:,}

Write a concise 4–5 sentence summary that:
1. Explains what these numbers mean in plain English.
2. Highlights the model's strengths and any limitations to watch for.
3. Suggests one concrete next step to improve the system further.
Do NOT use bullet points — write flowing prose.
"""

        response = self.client.messages.create(
            model=self.MODEL_ID,
            max_tokens=self.MAX_TOKENS,
            temperature=0.5,
            messages=[{"role": "user", "content": prompt}],
        )

        summary = response.content[0].text.strip()
        self._log("Summary received.\n")
        return summary

In [ ]:
# Orchestrator  — wire the agents together
def run_pipeline():
    print("=" * 60)
    print("  FAKE NEWS DETECTION — MULTI-AGENT PIPELINE")
    print("=" * 60 + "\n")

    # Step 1 – ingest
    ingestion_agent = IngestionAgent()
    df_raw = ingestion_agent.run()

    # Step 2 – feature engineering + EDA
    feature_agent = FeatureAgent()
    df_features = feature_agent.run(df_raw)

    # Step 3 – train & evaluate
    classifier_agent = ClassifierAgent()
    metrics = classifier_agent.run(df_features)

    # Step 4 – LLM narrative
    insight_agent = InsightAgent(anthropic_client)
    narrative = insight_agent.run(metrics)

    print("\n" + "─" * 60)
    print("  CLAUDE'S ANALYSIS")
    print("─" * 60)
    print(narrative)
    print("─" * 60 + "\n")

    # Quick inference demo
    headlines = [
        "Scientists confirm breakthrough cancer vaccine shows 90% efficacy in trials",
        "SHOCKING: Government secretly puts 5G chips in tap water to control minds!!!",
        "Local council approves new cycling lanes in city centre",
    ]

    print("  INFERENCE DEMO")
    print("─" * 60)
    model = classifier_agent.model
    for headline in headlines:
        pred  = model.predict([headline])[0]
        proba = model.predict_proba([headline])[0]
        label = "REAL" if pred == 1 else "FAKE"
        conf  = max(proba) * 100
        print(f"  [{label} — {conf:.1f}% confidence]")
        print(f"  \"{headline}\"\n")



In [ ]:
run_pipeline()